Visualization of skewness, mean and entropy on original images 

In [1]:
import os
from pathlib import Path
import numpy as np
import cv2
import matplotlib.pyplot as plt
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from scipy.stats import skew
from skimage.measure import shannon_entropy

# Confirmed BTT checkpoint from step 0.
model_path = '/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/BTTypes_MobileNetV2.keras'
loaded_model = load_model(model_path)

last_conv_layer_name = 'Conv_1_bn'
dense_layer_name = 'dense_1'
conv_layer_model = Model(
    inputs=loaded_model.input,
    outputs=loaded_model.get_layer(last_conv_layer_name).output
)
dense_layer_model = Model(
    inputs=loaded_model.input,
    outputs=loaded_model.get_layer(dense_layer_name).output
)

# Sample BTT images from both class folders; no masks are used.
image_dir = Path('/Users/kaivalyapuranik/Downloads/archive')
output_dir = Path('/Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/heatmap_overlays_BTTypes')
output_dir.mkdir(parents=True, exist_ok=True)
input_size = (224, 224)
sample_limit_per_class = 3
image_extensions = {'.png', '.jpg', '.jpeg'}

sample_images = []
for class_dir in sorted(path for path in image_dir.iterdir() if path.is_dir()):
    class_images = sorted(
        path for path in class_dir.iterdir()
        if path.is_file() and path.suffix.lower() in image_extensions
    )[:sample_limit_per_class]
    sample_images.extend((class_dir.name, image_path) for image_path in class_images)


def process_image(class_name, image_path):
    original_image = load_img(image_path, target_size=input_size)
    image_array = img_to_array(original_image)
    model_input = preprocess_input(np.expand_dims(image_array.copy(), axis=0))

    conv_features = conv_layer_model.predict(model_input, verbose=0).squeeze()
    dense_features = dense_layer_model.predict(model_input, verbose=0).squeeze()

    mean_map = np.zeros(conv_features.shape[:2])
    skewness_map = np.zeros(conv_features.shape[:2])
    entropy_map = np.zeros(conv_features.shape[:2])
    for channel_index in range(conv_features.shape[-1]):
        feature_map = conv_features[:, :, channel_index]
        mean_map += feature_map / conv_features.shape[-1]
        skewness_map += skew(feature_map.ravel()) * feature_map
        entropy_map += shannon_entropy(feature_map) * feature_map

    def normalize(feature_map):
        return (feature_map - feature_map.min()) / (
            feature_map.max() - feature_map.min() + 1e-8
        )

    mean_map = normalize(mean_map)
    skewness_map = normalize(skewness_map)
    entropy_map = normalize(entropy_map)

    dense_mean = np.mean(dense_features)
    dense_skewness = skew(dense_features.ravel())
    dense_entropy = shannon_entropy(dense_features)
    combined_map = (
        dense_mean * mean_map
        + dense_skewness * skewness_map
        + dense_entropy * entropy_map
    )
    combined_map = normalize(combined_map)

    mean_map_resized = cv2.resize(mean_map, (input_size[1], input_size[0]))
    skewness_map_resized = cv2.resize(skewness_map, (input_size[1], input_size[0]))
    entropy_map_resized = cv2.resize(entropy_map, (input_size[1], input_size[0]))
    combined_map_resized = cv2.resize(combined_map, (input_size[1], input_size[0]))

    figure, axes = plt.subplots(1, 5, figsize=(22, 6))
    axes[0].imshow(original_image)
    axes[0].set_title(f'{class_name}: Original')
    axes[1].imshow(original_image)
    axes[1].imshow(mean_map_resized, cmap='jet', alpha=0.5)
    axes[1].set_title('Mean Heatmap')
    axes[2].imshow(original_image)
    axes[2].imshow(skewness_map_resized, cmap='jet', alpha=0.5)
    axes[2].set_title('Skewness Heatmap')
    axes[3].imshow(original_image)
    axes[3].imshow(entropy_map_resized, cmap='jet', alpha=0.5)
    axes[3].set_title('Entropy Heatmap')
    axes[4].imshow(original_image)
    axes[4].imshow(combined_map_resized, cmap='jet', alpha=0.5)
    axes[4].set_title('Dense-weighted Combined')

    for axis in axes:
        axis.axis('off')
    figure.tight_layout()

    output_name = f'{class_name}_{image_path.stem}_heatmaps.png'
    output_path = output_dir / output_name
    figure.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(figure)
    print(f'Saved: {output_path}')


for class_name, image_path in sample_images:
    print(f'Processing {class_name}/{image_path.name}...')
    process_image(class_name, image_path)

print(f'Generated {len(sample_images)} mask-free heatmap overlays in {output_dir}')

Processing Benign/B (1).png...
Saved: /Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/heatmap_overlays_BTTypes/Benign_B (1)_heatmaps.png
Processing Benign/B (10).png...
Saved: /Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/heatmap_overlays_BTTypes/Benign_B (10)_heatmaps.png
Processing Benign/B (100).png...
Saved: /Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/heatmap_overlays_BTTypes/Benign_B (100)_heatmaps.png
Processing Malignant/M (1).png...
Saved: /Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/heatmap_overlays_BTTypes/Malignant_M (1)_heatmaps.png
Processing Malignant/M (10).png...
Saved: /Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/heatmap_overlays_BTTypes/Malignant_M (10)_heatmaps.png
Processing Malignant/M (100).png...
Saved: /Users/kaivalyapuranik/Desktop/XAI-Med-Images-Stat-Visual-Rules/heatmap_overlays_BTTypes/Malignant_M (100)_heatmaps.png
Generated 6 mask-free heatmap overlays in /Users/kai